In [6]:
import os
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.embeddings import OllamaEmbeddings
from langchain.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter

CSV_FILE_PATH = r"C:\Users\Zygim\Downloads\archive\weather_data.csv"
OLLAMA_MODEL = "llama2:latest"


def load_and_process_csv(file_path):
    """Loads data from a CSV, splits it into chunks, and creates embedings.

    Args:
        file_path: The path to the CSV file.

    Returns:
        A FAISS vectorstore containing the embedded data. Returns None on error.
    """
    try:
        loader = CSVLoader(file_path)
        docs = loader.load()

        text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        split_docs = text_splitter.split_documents(docs)

        embeddings = OllamaEmbeddings(model=OLLAMA_MODEL)
        vectorstore = FAISS.from_documents(split_docs, embeddings)
        return vectorstore

    except FileNotFoundError:
        print(f"Error: CSV file not found at {file_path}")
        return None
    except Exception as e:
        print(f"An error occurred during data loading: {e}")
        return None




prompt_template = """
You are a helpful assistant that analyzes weather data.
Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer as concise as possible.

{context}

Question: {question}
Answer:
"""

def create_llm_chain(vectorstore):
    """Creates an LLMChain for querying the weather data.

    Args:
        vectorstore: The FAISS vectorstore containing the embedded data.

    Returns:
        An LLMChain object, or None on error.
    """
    try:
        llm = Ollama(model=OLLAMA_MODEL)
        prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
        chain = LLMChain(prompt=prompt, llm=llm)

        retriever = vectorstore.as_retriever()

        return chain, retriever

    except Exception as e:
        print(f"An error occurred during LLM initialization: {e}")
        return None, None



def main():
    """Loads the data, sets up the LLM, and allows the user to ask questions."""

    vectorstore = load_and_process_csv(CSV_FILE_PATH)
    if vectorstore is None:
        return

    llm_chain, retriever = create_llm_chain(vectorstore)
    if llm_chain is None:
        return

    while True:
        question = input("Ask a question about the weather data (or type 'exit' to quit): ")
        if question.lower() == 'exit':
            break

        try:
            docs = retriever.get_relevant_documents(question)
            context = "\n".join([doc.page_content for doc in docs])
            response = llm_chain.run({"context": context, "question": question})
            print(response)

        except Exception as e:
            print(f"An error occurred while processing your question: {e}")



if __name__ == "__main__":
    main()


KeyboardInterrupt

